# Deploy Jina Embeddings v4 with vLLM

This notebook deploys Jina Embeddings v4 model using vLLM for high-performance inference

## 1. Install Dependencies

In [0]:
# Install required packages
%pip install vllm==0.7.0 transformers==4.53.0 torch accelerate==1.8.1 pandas tiktoken==0.9.0 Pillow requests
dbutils.library.restartPython()

  Obtaining dependency information for vllm==0.7.0 from https://files.pythonhosted.org/packages/51/70/6fc00dca2e9f53a76b7792d788cb2efbb9d2587ed0ca9a71d5ccf7fc7543/vllm-0.7.0-cp38-abi3-manylinux1_x86_64.whl.metadata
  Obtaining dependency information for transformers==4.53.0 from https://files.pythonhosted.org/packages/5e/0c/68d03a38f6ab2ba2b2829eb11b334610dd236e7926787f7656001b68e1f2/transformers-4.53.0-py3-none-any.whl.metadata
  Obtaining dependency information for accelerate==1.8.1 from https://files.pythonhosted.org/packages/91/d9/e044c9d42d8ad9afa96533b46ecc9b7aea893d362b3c52bd78fb9fe4d7b3/accelerate-1.8.1-py3-none-any.whl.metadata
  Obtaining dependency information for tiktoken==0.9.0 from https://files.pythonhosted.org/packages/b1/73/41591c525680cd460a6becf56c9b17468d3711b1df242c53d2c7b2183d16/tiktoken-0.9.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata
  Obtaining dependency information for blake3 from https://files.pythonhosted.org/packages/f6/58/16b74904

## 2. Import Libraries

In [0]:
import os
import json
import time
import uuid
import mlflow
import pandas as pd
from typing import List, Dict, Any, Optional
import logging
from datetime import timedelta
from databricks.sdk import WorkspaceClient
from databricks.sdk.runtime import dbutils
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    AutoCaptureConfigInput,
    ServedEntityInput,
    TrafficConfig,
    Route,
    EndpointTag,
    ServingEndpointAccessControlRequest,
    ServingEndpointPermissionLevel,
)
from databricks.sdk.errors import ResourceDoesNotExist

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

## 3. Configuration

In [0]:
# Model configuration
MODEL_NAME = "jinaai/jina-embeddings-v4-vllm-retrieval" 
REGISTERED_MODEL_NAME = "jina-embeddings-v4-vllm-retrieval"

# Environment setup
ENVIRONMENT = "dev"  # Options: dev, uat, prod
USERNAME = "hoang.pham.uh@renesas.com"

# Unity Catalog configuration
CATALOG = "ai_dev"
SCHEMA = "ai_dev_common_models"
FULL_MODEL_NAME = f"{CATALOG}.{SCHEMA}.{REGISTERED_MODEL_NAME}"

# MLflow setup
mlflow.set_registry_uri("databricks-uc")

INFO:py4j.clientserver:Received command c on object id p1
INFO:py4j.clientserver:Received command c on object id p0


## 4. Prepare PyFunc Wrapper Path

In [0]:
# Assuming wrapper is in current directory
model_path = os.path.join(os.getcwd(), "./Jina_wrapper.py")

# Verify file exists
if not os.path.exists(model_path):
    raise FileNotFoundError(f"Wrapper file not found at {model_path}")

print(f"Using wrapper at: {model_path}")

INFO:py4j.clientserver:Received command c on object id p1


Using wrapper at: /Workspace/Users/hoang.pham.uh@renesas.com/deploy_ColPali/./Jina_wrapper.py


## 5. Create Input/Output Examples and Signature

In [0]:
# Input examples for embeddings

# Example 1: Text only
input_example_text = {
    "texts": ["Hello world", "Machine learning is amazing"],
}

# Example 2: Image only (base64)
input_example_image = {
    "images": ["data:image/jpeg;base64,/9j/4AAQSkZJRg..."]  # Shortened for example
}

# Example 3: Multimodal (text + image)
input_example_multimodal = {
    "texts": ["A beautiful landscape"],
    "images": ["https://example.com/image.jpg"]
}

# Use text example as primary for signature
input_example = input_example_text

# Output example
output_example = {
    "model": "jinaai/jina-embeddings-v4-vllm-retrieval",
    "object": "list",
    "data": [
        {
            "object": "embedding",
            "embedding": [0.1, 0.2, 0.3],  # Simplified for example
            "index": 0
        },
        {
            "object": "embedding",
            "embedding": [0.4, 0.5, 0.6],
            "index": 1
        }
    ],
    "usage": {
        "prompt_tokens": 8,
        "total_tokens": 8
    },
    "id": "emb_abc123",
    "created": 1234567890,
    "performance": {
        "generation_time_ms": 50,
        "embeddings_per_second": 40.0,
        "tokens_per_second": 160.0,
        "backend": "vLLM"
    },
    "metadata": {
        "embedding_dimensions": 1024,
        "num_embeddings": 2,
        "input_type": "text",
        "num_texts": 2,
        "num_images": 0,
        "task": None,
        "normalized": True
    }
}

# Infer signature
from mlflow.models import infer_signature
signature = infer_signature(input_example, output_example)

print("Signature created successfully")
print(f"Supports: Text, Image, and Multimodal inputs")

INFO:py4j.clientserver:Received command c on object id p1


Signature created successfully


/databricks/python/lib/python3.11/site-packages/mlflow/types/utils.py:435: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


## 6. Log Model to MLflow

In [0]:
# Start MLflow run
with mlflow.start_run(run_name=f"{REGISTERED_MODEL_NAME}_run") as run:
    mlflow.set_tag("ai_project_id", "coding")
    mlflow.set_tag("model_type", "embedding")
    mlflow.set_tag("backend", "vLLM")
    mlflow.set_tag("multimodal", "true")
    
    model_info = mlflow.pyfunc.log_model(
        python_model=model_path,
        registered_model_name=FULL_MODEL_NAME,
        artifact_path=f"model/{REGISTERED_MODEL_NAME}",
        input_example=input_example,
        signature=signature,
        pip_requirements=[
            "vllm>=0.7.0",
            "transformers>=4.53.0",
            "torch>=2.5.0",
            "accelerate>=1.8.0",
            "pandas>=1.5.0",
            "numpy>=1.23.0",
            "Pillow>=10.0.0",
            "requests>=2.31.0"
        ],
    )
    
print("✅ Model logged successfully to MLflow")
print(f"Model URI: {model_info.model_uri}")
print(f"Run ID: {run.info.run_id}")
print(f"✅ Supports: Text, Image, and Multimodal embeddings")

INFO:py4j.clientserver:Received command c on object id p1
INFO:py4j.clientserver:Closing down clientserver connection
INFO:py4j.clientserver:Closing down clientserver connection
INFO:py4j.clientserver:Error while sending or receiving.
Traceback (most recent call last):
  File "/databricks/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 528, in send_command
    self.socket.sendall(command.encode("utf-8"))
ConnectionResetError: [Errno 104] Connection reset by peer
INFO:py4j.clientserver:Closing down clientserver connection
INFO:root:Exception while sending command.
Traceback (most recent call last):
  File "/databricks/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 528, in send_command
    self.socket.sendall(command.encode("utf-8"))
ConnectionResetError: [Errno 104] Connection reset by peer

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/databricks/spark/python/lib/py4j-0.10.9.7-src.

INFO 10-08 09:05:08 __init__.py:183] Automatically detected platform cuda.


INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:code_model_e796bfac09d846708fbf251bc02eef38:Loading Jina embedding model with vLLM: jinaai/jina-embeddings-v4-vllm-retrieval


config.json: 0.00B [00:00, ?B/s]

INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0


...

INFO:py4j.clientserver:Received command c on object id p0
Successfully registered model 'ai_dev.ai_dev_common_models.jina-embeddings-v4-vllm-retrieval'.
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0


Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

INFO:azure.core.pipeline.policies.http_logging_policy:Request URL: 'https://dlsqwestus2014.dfs.core.windows.net/ai-uat-model-repository/dev%2Fmodels%2Fc39497cf-2f17-4c5f-b56a-6286abc10f61%2Fversions%2F96672e5e-d547-43ab-8c30-0e4fa94e87e8%2Fconda.yaml?resource=REDACTED&st=REDACTED&sv=REDACTED&ske=REDACTED&sig=REDACTED&sktid=REDACTED&se=REDACTED&sdd=REDACTED&skoid=REDACTED&spr=REDACTED&sks=REDACTED&skt=REDACTED&sp=REDACTED&skv=REDACTED&sr=REDACTED'
Request method: 'PUT'
Request headers:
    'x-ms-version': 'REDACTED'
    'Accept': 'application/json'
    'User-Agent': 'azsdk-python-storage-dfs/12.14.0 Python/3.11.11 (Linux-5.15.0-1092-azure-x86_64-with-glibc2.35)'
    'x-ms-date': 'REDACTED'
    'x-ms-client-request-id': 'e7f1a9b6-a425-11f0-9f38-00163eb7b27c'
No body was attached to the request
INFO:azure.core.pipeline.policies.http_logging_policy:Request URL: 'https://dlsqwestus2014.dfs.core.windows.net/ai-uat-model-repository/dev%2Fmodels%2Fc39497cf-2f17-4c5f-b56a-6286abc10f61%2Fversion

🏃 View run jina-embeddings-v4-vllm-retrieval_run at: https://adb-379144824042062.2.azuredatabricks.net/ml/experiments/150327785206899/runs/555617ca68d44297a2684fab8210351f
🧪 View experiment at: https://adb-379144824042062.2.azuredatabricks.net/ml/experiments/150327785206899
✅ Model logged successfully to MLflow
Model URI: runs:/555617ca68d44297a2684fab8210351f/model/jina-embeddings-v4-vllm-retrieval
Run ID: 555617ca68d44297a2684fab8210351f


INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Closing down clientserver connection
INFO:py4j.clientserver:Closing down clientserver connection


## 7. Register Model to Unity Catalog

In [0]:
UC_MODEL_NAME = FULL_MODEL_NAME

print(f"Registering model to Unity Catalog: {UC_MODEL_NAME}")

uc_registered_model_info = mlflow.register_model(
    model_uri=model_info.model_uri, 
    name=UC_MODEL_NAME
)

print(f"✅ Model registered successfully")
print(f"Model Name: {uc_registered_model_info.name}")
print(f"Version: {uc_registered_model_info.version}")

INFO:py4j.clientserver:Received command c on object id p1
INFO:py4j.clientserver:Error while sending or receiving.
Traceback (most recent call last):
  File "/databricks/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 528, in send_command
    self.socket.sendall(command.encode("utf-8"))
ConnectionResetError: [Errno 104] Connection reset by peer
INFO:py4j.clientserver:Closing down clientserver connection
INFO:root:Exception while sending command.
Traceback (most recent call last):
  File "/databricks/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 528, in send_command
    self.socket.sendall(command.encode("utf-8"))
ConnectionResetError: [Errno 104] Connection reset by peer

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/databricks/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^

Registering model to Unity Catalog: ai_dev.ai_dev_common_models.jina-embeddings-v4-vllm-retrieval


Registered model 'ai_dev.ai_dev_common_models.jina-embeddings-v4-vllm-retrieval' already exists. Creating a new version of this model...
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Error while sending or receiving.
Traceback (most recent call last):
  File "/databricks/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 528, in send_command
    self.socket.sendall(command.encode("utf-8"))
ConnectionResetError: [Errno 104] Connection reset by peer
INFO:py4j.clientserver:Closing down clientserver connection
INFO:root:Exception while sending command.
Traceback (most recent call last):
  File "/databricks/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 528, in send_command
    self.socket.sendall(command.encode("utf-8"))
ConnectionResetError: [Errno 104] Connection reset by peer

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/databricks/spark/python/lib/p

INFO:py4j.clientserver:Received command c on object id p0


Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

INFO:azure.core.pipeline.policies.http_logging_policy:Request URL: 'https://dlsqwestus2014.dfs.core.windows.net/ai-uat-model-repository/dev%2Fmodels%2Fc39497cf-2f17-4c5f-b56a-6286abc10f61%2Fversions%2F9385a59d-4e33-499a-b2bf-e5ef86c80305%2Fconda.yaml?resource=REDACTED&st=REDACTED&sv=REDACTED&ske=REDACTED&sig=REDACTED&sktid=REDACTED&se=REDACTED&sdd=REDACTED&skoid=REDACTED&spr=REDACTED&sks=REDACTED&skt=REDACTED&sp=REDACTED&skv=REDACTED&sr=REDACTED'
Request method: 'PUT'
Request headers:
    'x-ms-version': 'REDACTED'
    'Accept': 'application/json'
    'User-Agent': 'azsdk-python-storage-dfs/12.14.0 Python/3.11.11 (Linux-5.15.0-1092-azure-x86_64-with-glibc2.35)'
    'x-ms-date': 'REDACTED'
    'x-ms-client-request-id': 'eb978dce-a425-11f0-8ede-00163eb7b27c'
No body was attached to the request
INFO:azure.core.pipeline.policies.http_logging_policy:Request URL: 'https://dlsqwestus2014.dfs.core.windows.net/ai-uat-model-repository/dev%2Fmodels%2Fc39497cf-2f17-4c5f-b56a-6286abc10f61%2Fversion

✅ Model registered successfully
Model Name: ai_dev.ai_dev_common_models.jina-embeddings-v4-vllm-retrieval
Version: 2


Created version '2' of model 'ai_dev.ai_dev_common_models.jina-embeddings-v4-vllm-retrieval'.


## 8. Configure Endpoint Settings

In [0]:
# Endpoint configuration based on environment
scale_to_zero_enabled = True if ENVIRONMENT in ("dev", "uat") else False

ENDPOINT_NAME = UC_MODEL_NAME.split(".")[-1]

# Workload size based on environment
if ENVIRONMENT == "dev":
    workload_size = "Small"
    print("Using Small workload size for dev environment")
elif ENVIRONMENT == "uat":
    workload_size = "Medium"
    print("Using Medium workload size for UAT environment")
else:
    workload_size = "Large"
    print("Using Large workload size for production")

# Configure served entities
served_entities = [
    ServedEntityInput(
        name=ENDPOINT_NAME,
        entity_name=uc_registered_model_info.name,
        entity_version=uc_registered_model_info.version,
        workload_type="GPU_SMALL",  # T4 GPU for embeddings
        workload_size=workload_size,
        scale_to_zero_enabled=scale_to_zero_enabled
    )
]

# Traffic configuration
traffic_config = TrafficConfig([
    Route(served_model_name=ENDPOINT_NAME, traffic_percentage=100)
])

print(f"Endpoint configuration:")
print(f"  - Name: {ENDPOINT_NAME}")
print(f"  - Workload: GPU_SMALL ({workload_size})")
print(f"  - Scale to Zero: {scale_to_zero_enabled}")

Using Small workload size for dev environment
Endpoint configuration:
  - Name: jina-embeddings-v4-vllm-retrieval
  - Workload: GPU_SMALL (Small)
  - Scale to Zero: True


## 9. Create/Update Serving Endpoint

In [0]:
w = WorkspaceClient()

endpoint_config = EndpointCoreConfigInput(
    served_entities=served_entities,
    traffic_config=traffic_config,
)

try:
    # Check if endpoint exists
    endpoint = w.serving_endpoints.get(ENDPOINT_NAME)
    print(f"📝 Endpoint '{ENDPOINT_NAME}' exists. Updating...")
    print(f"\n🔗 View deployment: https://adb-379144824042062.2.azuredatabricks.net/ml/endpoints/{ENDPOINT_NAME}")
    
    # Update existing endpoint
    w.serving_endpoints.update_config_and_wait(
        name=ENDPOINT_NAME,
        served_entities=served_entities,
        traffic_config=traffic_config,
        timeout=timedelta(minutes=30),
    )
    
    # Update tags
    try:
        w.serving_endpoints.patch(
            name=ENDPOINT_NAME,
            add_tags=[
                EndpointTag(key="ai_project_id", value="embeddings"),
                EndpointTag(key="model_type", value="embedding"),
                EndpointTag(key="backend", value="vLLM")
            ],
        )
    except AttributeError as e:
        print(f"⚠️  Warning: Tags updated but with warning: {e}")
    
    print(f"✅ Endpoint '{ENDPOINT_NAME}' updated successfully")
    
except ResourceDoesNotExist:
    # Create new endpoint
    print(f"🆕 Endpoint does not exist. Creating new endpoint '{ENDPOINT_NAME}'...")
    print(f"\n🔗 View deployment: https://adb-379144824042062.2.azuredatabricks.net/ml/endpoints/{ENDPOINT_NAME}")
    
    w.serving_endpoints.create_and_wait(
        name=ENDPOINT_NAME,
        config=endpoint_config,
        tags=[
            EndpointTag(key="ai_project_id", value="embeddings"),
            EndpointTag(key="model_type", value="embedding"),
            EndpointTag(key="backend", value="vLLM")
        ],
        timeout=timedelta(minutes=30),
    )
    
    print(f"✅ Endpoint '{ENDPOINT_NAME}' created successfully")

INFO:py4j.clientserver:Received command c on object id p1


🆕 Endpoint does not exist. Creating new endpoint 'jina-embeddings-v4-vllm-retrieval'...

🔗 View deployment: https://adb-379144824042062.2.azuredatabricks.net/ml/endpoints/jina-embeddings-v4-vllm-retrieval


INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clie

---------------------------------------------------------------------------
ResourceDoesNotExist                      Traceback (most recent call last)
File <command-8721849208413917>, line 10
      8 try:
      9     # Check if endpoint exists
---> 10     endpoint = w.serving_endpoints.get(ENDPOINT_NAME)
     11     print(f"📝 Endpoint '{ENDPOINT_NAME}' exists. Updating...")

File /databricks/python/lib/python3.11/site-packages/databricks/sdk/service/serving.py:2521, in ServingEndpointsAPI.get(self, name)
   2519 headers = {'Accept': 'application/json', }
-> 2521 res = self._api.do('GET', f'/api/2.0/serving-endpoints/{name}', headers=headers)
   2522 return ServingEndpointDetailed.from_dict(res)

File /databricks/python/lib/python3.11/site-packages/databricks/sdk/core.py:130, in ApiClient.do(self, method, path, query, headers, body, raw, files, data, response_headers)
    127 retryable = retried(timeout=timedelta(seconds=self._retry_timeout_seconds),
    128                     is_retr

## 10. Set Endpoint Permissions

In [0]:
try:
    # Get endpoint details
    endpoint = w.serving_endpoints.get(name=ENDPOINT_NAME)
    serving_endpoint_id = endpoint.id
    
    # Set permissions for team members
    w.serving_endpoints.set_permissions(
        serving_endpoint_id=serving_endpoint_id,
        access_control_list=[
            ServingEndpointAccessControlRequest(
                user_name="hoang.pham.uh@renesas.com",
                permission_level=ServingEndpointPermissionLevel.CAN_MANAGE,
            ),
            ServingEndpointAccessControlRequest(
                user_name="dung.nguyen.te@renesas.com",
                permission_level=ServingEndpointPermissionLevel.CAN_MANAGE,
            )
        ],
    )
    
    print("✅ Permissions set successfully")
    
except Exception as e:
    print(f"⚠️  Error setting permissions: {e}")

# End MLflow run
mlflow.end_run()

print("\n" + "="*60)
print("🎉 DEPLOYMENT COMPLETED SUCCESSFULLY!")
print("="*60)
print(f"\n📊 Endpoint Details:")
print(f"   - Name: {ENDPOINT_NAME}")
print(f"   - Model: {FULL_MODEL_NAME}")
print(f"   - Version: {uc_registered_model_info.version}")
print(f"   - Environment: {ENVIRONMENT}")
print(f"\n🔗 Access endpoint at:")
print(f"   https://adb-379144824042062.2.azuredatabricks.net/ml/endpoints/{ENDPOINT_NAME}")

---------------------------------------------------------------------------
ResourceDoesNotExist                      Traceback (most recent call last)
File <command-8721849208413917>, line 10
      8 try:
      9     # Check if endpoint exists
---> 10     endpoint = w.serving_endpoints.get(ENDPOINT_NAME)
     11     print(f"📝 Endpoint '{ENDPOINT_NAME}' exists. Updating...")

File /databricks/python/lib/python3.11/site-packages/databricks/sdk/service/serving.py:2521, in ServingEndpointsAPI.get(self, name)
   2519 headers = {'Accept': 'application/json', }
-> 2521 res = self._api.do('GET', f'/api/2.0/serving-endpoints/{name}', headers=headers)
   2522 return ServingEndpointDetailed.from_dict(res)

File /databricks/python/lib/python3.11/site-packages/databricks/sdk/core.py:130, in ApiClient.do(self, method, path, query, headers, body, raw, files, data, response_headers)
    127 retryable = retried(timeout=timedelta(seconds=self._retry_timeout_seconds),
    128                     is_retr

## 11. Test Endpoint (Optional)

In [0]:
# Test the deployed endpoint
import requests
import os

# Get Databricks token
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

# Endpoint URL
endpoint_url = f"https://adb-379144824042062.2.azuredatabricks.net/serving-endpoints/{ENDPOINT_NAME}/invocations"

headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json"
}

print("=" * 60)
print("TESTING ENDPOINT")
print("=" * 60)

# Test 1: Text embeddings
print("\n📝 Test 1: Text Embeddings")
test_payload_text = {
    "inputs": {
        "texts": ["What is machine learning?", "How does AI work?"]
    }
}

try:
    response = requests.post(endpoint_url, json=test_payload_text, headers=headers)
    
    if response.status_code == 200:
        result = response.json()
        print("✅ Text embedding test successful!")
        print(f"   - Embeddings generated: {result['metadata']['num_embeddings']}")
        print(f"   - Dimensions: {result['metadata']['embedding_dimensions']}")
        print(f"   - Time: {result['performance']['generation_time_ms']}ms")
    else:
        print(f"❌ Test failed with status {response.status_code}")
        print(f"Error: {response.text}")
        
except Exception as e:
    print(f"❌ Error testing endpoint: {e}")

# Test 2: Image embeddings (optional - uncomment if you have an image URL)
print("\n🖼️ Test 2: Image Embeddings (Optional)")
print("   To test image embeddings, use:")
print("   test_payload_image = {")
print('       "inputs": {')
print('           "images": ["https://example.com/image.jpg"]')
print("       }")
print("   }")

# Test 3: Show multimodal example
print("\n🎨 Test 3: Multimodal Example")
print("   To test multimodal embeddings, use:")
print("   test_payload_multimodal = {")
print('       "inputs": {')
print('           "texts": ["A beautiful sunset over the ocean"],')
print('           "images": ["https://example.com/sunset.jpg"]')
print("       }")
print("   }")

---------------------------------------------------------------------------
ResourceDoesNotExist                      Traceback (most recent call last)
File <command-8721849208413917>, line 10
      8 try:
      9     # Check if endpoint exists
---> 10     endpoint = w.serving_endpoints.get(ENDPOINT_NAME)
     11     print(f"📝 Endpoint '{ENDPOINT_NAME}' exists. Updating...")

File /databricks/python/lib/python3.11/site-packages/databricks/sdk/service/serving.py:2521, in ServingEndpointsAPI.get(self, name)
   2519 headers = {'Accept': 'application/json', }
-> 2521 res = self._api.do('GET', f'/api/2.0/serving-endpoints/{name}', headers=headers)
   2522 return ServingEndpointDetailed.from_dict(res)

File /databricks/python/lib/python3.11/site-packages/databricks/sdk/core.py:130, in ApiClient.do(self, method, path, query, headers, body, raw, files, data, response_headers)
    127 retryable = retried(timeout=timedelta(seconds=self._retry_timeout_seconds),
    128                     is_retr

## Summary

This notebook:
1. ✅ Installs required dependencies (including Pillow for image support)
2. ✅ Loads Jina Embeddings v4 multimodal wrapper
3. ✅ Logs model to MLflow with proper signature
4. ✅ Registers model to Unity Catalog
5. ✅ Creates/updates serving endpoint with vLLM
6. ✅ Sets appropriate permissions
7. ✅ Tests the deployed endpoint

### Features:
- 🔤 **Text Embeddings**: Generate embeddings for text inputs
- 🖼️ **Image Embeddings**: Generate embeddings for images (base64, URL, file path)
- 🎨 **Multimodal**: Generate embeddings for both text and images together
- ⚡ **vLLM Optimized**: High-performance inference with GPU acceleration

### Usage Examples:

#### Text Only:
```python
payload = {
    "inputs": {
        "texts": ["Your text here", "Another text"]
    }
}
```

#### Image Only:
```python
payload = {
    "inputs": {
        "images": [
            "https://example.com/image.jpg",
            "data:image/jpeg;base64,/9j/4AAQSkZJRg..."
        ]
    }
}
```

#### Multimodal (Text + Image):
```python
payload = {
    "inputs": {
        "texts": ["A description of the image"],
        "images": ["https://example.com/image.jpg"]
    }
}
```

### API Call:
```python
import requests

response = requests.post(endpoint_url, json=payload, headers=headers)
embeddings = response.json()
```